# Кейс 1: Рекомендации банковских продуктов  
**Цель:** предсказать, какой банковский продукт стоит предложить клиенту.  
**Основные задачи:** анализ событий покупок и характеристик клиентов, построение ALS-признака, обучение многоклассовой модели, экспорт артефактов для веб-сервиса и мониторинг качества сервиса.

---
**Актуальная методология:**
- статистики предобработки (медианы, моды, квантили клиппинга, редкие категории,
  возрастные бины, когорты дат) обучаются только на train-части;
- разбиение train/test — временное по дате привлечения клиента `fecha_alta`:
  train содержит более ранние когорты, test — более новые;
- ALS оценивается в двух скоупах: на клиентах, присутствующих в 2015 и 2016 годах,
  и на всех покупателях 2016 года, включая cold-start;
- качество финальной рекомендации читается по per-class метрикам, macro-метрикам
  и PR-AUC/lift относительно случайного бейзлайна, а не только по accuracy.


In [37]:
# Стандартная библиотека
import gc
import json
import logging
import os
import warnings

# Сторонние библиотеки
from dotenv import load_dotenv
from implicit.als import AlternatingLeastSquares
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import scipy
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tqdm
from sklearn.preprocessing import label_binarize

In [2]:
gc.collect()


20

In [3]:
# Датасет скачивается из соревнования Kaggle ноутбуком loader.ipynb
df = pd.read_csv('data/train_ver2.csv')
df


C:\Users\Admin\AppData\Local\Temp\ipykernel_2896\2741283767.py:2: DtypeWarning: Columns (5,8,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/train_ver2.csv')


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.0,6,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13647304,2016-05-28,1166765,N,ES,V,22,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647305,2016-05-28,1166764,N,ES,V,23,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647306,2016-05-28,1166763,N,ES,H,47,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647307,2016-05-28,1166789,N,ES,H,22,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0


In [4]:
# Единый список продуктов и функции предобработки — из общего модуля
# обучение и сервис используют один и тот же код.
# Код целевой переменной `purchase` = позиция продукта в списке + 1.
from preprocessing import (
    BASE_COLUMNS,
    PRODUCTS as products,
    apply_preprocessing,
    feature_engineering,
    fit_preprocessing_params,
    temporal_split,
)


In [5]:
# Удаляем малоинформативные столбцы: много пропусков, дублирование других
# признаков или отсутствие в итоговой матрице признаков модели.
df = df.drop(columns=['indrel', 'indext', 'nomprov', 'tipodom'])


In [6]:
# преобразуем столбцы в числовые типы данных
quantitative_cols = ['age', 'antiguedad', 'renta']
for col in quantitative_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# заменяем аномальное значение в antiguedad на NaN
df['antiguedad'] = df['antiguedad'].replace(np.float64(-999999.0), np.nan)

# Преобразуем колонку fecha_dato в формат datetime
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])


# Персональные
Рассчитаем персональные рекомендации.

In [7]:
# Целевая переменная строится по всем 24 продуктам.
# События покупок: продукт появился у клиента впервые (было 0 -> стало 1).
df_sorted = df.sort_values(['ncodpers', 'fecha_dato'])

# Кодируем клиентов и продукты в числовые индексы
users = list(df['ncodpers'].unique())
user_map = {int(user): idx for idx, user in enumerate(users)}
product_map = {product: idx for idx, product in enumerate(products)}

# Предыдущее состояние продукта у клиента — одним groupby на все продукты сразу
products_prev = df_sorted.groupby('ncodpers')[products].shift(1).fillna(0)
current = df_sorted[products].fillna(0).to_numpy(dtype='int8')
previous = products_prev.to_numpy(dtype='int8')
new_purchase = (current == 1) & (previous == 0)

rows, cols = np.nonzero(new_purchase)
events = pd.DataFrame({
    'ncodpers': df_sorted['ncodpers'].to_numpy()[rows],
    'fecha_dato': df_sorted['fecha_dato'].to_numpy()[rows],
    'purchase': (cols + 1).astype(int),  # код продукта = позиция в products + 1
})
print(f'Всего событий покупок: {len(events):,}')
print(events.head())

# Шорт-лист целевых продуктов: берем продукты, на которые пришлось не меньше
# MIN_PURCHASE_SHARE покупок 2015 года (обучающий период, без подглядывания в 2016-й).
# Осознанное сужение задачи: остальные продукты встречаются слишком редко,
# чтобы обучить на них устойчивый класс.
MIN_PURCHASE_SHARE = 0.01   # порог по доле покупок 2015 года
MAX_TARGET_PRODUCTS = 12    # ограничение на размер шорт-листа (размерность классов)
OTHER_CLASS = 99

train_purchases = events[events['fecha_dato'] <= '2016-01-01']
purchase_share = train_purchases['purchase'].value_counts(normalize=True)
candidates = purchase_share[purchase_share >= MIN_PURCHASE_SHARE]
target_codes = sorted(int(code) for code in candidates.index[:MAX_TARGET_PRODUCTS])
dropped_by_cap = [int(code) for code in candidates.index[MAX_TARGET_PRODUCTS:]]

label_map = {0: 'no_purchase'}
label_map.update({code: products[code - 1] for code in target_codes})
label_map[OTHER_CLASS] = 'other'

print(f'Продуктов всего: {len(products)}; прошли порог {MIN_PURCHASE_SHARE:.0%}: '
      f'{len(candidates)}; в шорт-листе: {len(target_codes)}')
for code in target_codes:
    print(f'  {code:>2} = {products[code - 1]:<22} доля покупок 2015: {purchase_share[code]:.2%}')
if dropped_by_cap:
    print(f'Отброшено ограничением MAX_TARGET_PRODUCTS={MAX_TARGET_PRODUCTS}: '
          f'{[products[code - 1] for code in dropped_by_cap]}')
print(f'Класс {OTHER_CLASS} = other: {1 - purchase_share[target_codes].sum():.2%} покупок 2015 '
      f'приходится на продукты вне шорт-листа')
print('label_map:', label_map)


Всего событий покупок: 1,778,496
   ncodpers fecha_dato  purchase
0     15889 2015-01-28         3
1     15889 2015-01-28         9
2     15889 2015-01-28        19
3     15889 2015-01-28        20
4     15889 2015-05-28        19
Продуктов всего: 24; прошли порог 1%: 13; в шорт-листе: 12
   3 = ind_cco_fin_ult1       доля покупок 2015: 39.17%
   5 = ind_cno_fin_ult1       доля покупок 2015: 5.40%
   8 = ind_ctop_fin_ult1      доля покупок 2015: 6.92%
   9 = ind_ctpp_fin_ult1      доля покупок 2015: 2.40%
  12 = ind_dela_fin_ult1      доля покупок 2015: 3.05%
  13 = ind_ecue_fin_ult1      доля покупок 2015: 4.78%
  18 = ind_reca_fin_ult1      доля покупок 2015: 2.87%
  19 = ind_tjcr_fin_ult1      доля покупок 2015: 5.37%
  20 = ind_valo_fin_ult1      доля покупок 2015: 1.50%
  22 = ind_nomina_ult1        доля покупок 2015: 5.53%
  23 = ind_nom_pens_ult1      доля покупок 2015: 6.22%
  24 = ind_recibo_ult1        доля покупок 2015: 12.55%
Отброшено ограничением MAX_TARGET_PRODUCTS=12: [

In [8]:
common_train = events[events['fecha_dato'] <= '2016-01-01'].copy()
common_test = events[events['fecha_dato'] > '2016-01-01'].copy()


In [9]:
# Создаём sparse-матрицу формата CSR.
# Строки матрицы — кодированные индексы user_map, и с теми же индексами
# вызывается ALS.recommend(). Это обеспечивает корректное сопоставление
# клиента и персональной рекомендации.
user_item_matrix_train = scipy.sparse.csr_matrix((
    np.ones(len(common_train)),
    (common_train['ncodpers'].map(user_map).to_numpy(),
     common_train['purchase'].to_numpy())),
    dtype=np.int8)


In [10]:
als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05,
                                    random_state=0)
als_model.fit(user_item_matrix_train)


c:\Users\Admin\miniconda3\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 20 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [01:06<00:00,  1.32s/it]


In [11]:
def get_recommendations_als(user_item_matrix, model, user_id, user_map, product_map,
                            include_seen=True, n=5):
    """Возвращает отранжированные ALS-рекомендации для заданного пользователя."""
    # Получаем закодированный user_id
    user_id_enc = user_map[user_id]

    # Создаем обратный словарь для product_map
    inv_product_map = {v: k for k, v in product_map.items()}

    # Получаем рекомендации от модели
    indices, scores = model.recommend(
        user_id_enc,
        user_item_matrix[user_id_enc],
        filter_already_liked_items=not include_seen,
        N=n
    )

    # Создаем итоговый DataFrame в pandas (polars в проекте не используется)
    return pd.DataFrame({
        'product': [inv_product_map.get(int(index)) for index in indices],
        'score': scores
    })


In [12]:
# Пересечение пользователей через set: быстрее и явно фиксирует скоуп оценки.
train_users = common_train['ncodpers'].unique()
test_users = set(common_test['ncodpers'].unique())
common_users = np.array([u for u in train_users if u in test_users])

# user_map — dict, вытаскиваем индексы напрямую
user_ids_encoded = np.array([user_map[int(x)] for x in common_users], dtype=np.int32)

# Батчевые рекомендации: ALS.recommend на списке крутит Python-цикл внутри,
# чанки + tqdm дают тот же результат и видимый прогресс
BATCH_SIZE = 2048
ids_parts, scores_parts = [], []
for start in tqdm.tqdm(range(0, len(user_ids_encoded), BATCH_SIZE), desc='ALS recommend'):
    batch = user_ids_encoded[start:start + BATCH_SIZE]
    batch_ids, batch_scores = als_model.recommend(
        batch,
        user_item_matrix_train[batch],
        filter_already_liked_items=True,
        N=1,
    )
    ids_parts.append(batch_ids)
    scores_parts.append(batch_scores)

# Контракт прежний (tuple), downstream-код менять не нужно
als_recommendations = (
    np.concatenate(ids_parts, axis=0),
    np.concatenate(scores_parts, axis=0),
)


ALS recommend: 100%|██████████| 46/46 [00:04<00:00, 10.20it/s]


In [13]:
# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(),
    "score": als_scores.tolist()})

# Разворачиваем (exploding) столбцы item_id_enc и score в отдельные строки.
# ignore_index=True сбрасывает индекс результирующего DataFrame,
# чтобы индексы были последовательными
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")
inv_product_map = {v: k for k, v in product_map.items()}
inv_user_map = {v: k for k, v in user_map.items()}
# получаем изначальные идентификаторы
als_recommendations["ncodpers"] = als_recommendations["user_id_enc"].map(inv_user_map)
als_recommendations["product"] = als_recommendations["item_id_enc"].map(inv_product_map)
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])


In [14]:
common_users_list = [int(x) for x in common_users]
filtered_events_test = common_test[common_test['ncodpers'].isin(common_users_list)]


In [15]:
recommended_products_list = []
user_ids_list = []
non_common_users = list(filtered_events_test['ncodpers'].unique())
for user in non_common_users:
    user_ids_list.extend([user])
    recommended_products_list.append(np.nan)
recommended_tracks_df = pd.DataFrame({'ncodpers': user_ids_list,
                                      'product': recommended_products_list})
als_recs = als_recommendations[['ncodpers', 'product']]
recommended_tracks_df = pd.concat([recommended_tracks_df, als_recs], axis=0)
recommended_tracks_df.rename(columns={'product': 'recommended_product_id'}, inplace=True)
# Один клиент мог попасть и в NaN-строки, и в ALS-рекомендации: при дедупе
# оставляем непустую рекомендацию (NaN сортируется последним, keep='first'
# берёт непустое). Иначе lookup в сервисе всегда находил бы NaN → 0.
recommended_tracks_df = recommended_tracks_df.sort_values('recommended_product_id')
recommended_tracks_df = recommended_tracks_df.drop_duplicates('ncodpers', keep='first')
print(f'Клиентов с персональной рекомендацией: '
      f"{int(recommended_tracks_df['recommended_product_id'].notna().sum())} "
      f'из {len(recommended_tracks_df)}')
recommended_tracks_df.set_index('ncodpers').to_parquet('fastapi/personal_als.parquet')


Клиентов с персональной рекомендацией: 93907 из 94091


### Качество ALS: precision@k / recall@k в двух скоупах

ALS используется как самостоятельный простой baseline и как источник признака
`recommended_product_id` для финального классификатора. Рекомендации строятся по
истории 2015 года; релевантными считаются продукты, которые клиент действительно
купил в 2016 году.

Скоупы:
- `common_users_2015_2016` — клиенты, присутствовавшие в обоих годах;
- `all_users_2016` — все клиенты с событиями 2016 года; для клиентов без истории
  2015 года персональных ALS-рекомендаций нет (hits = 0).

В последнем запуске top-5 ALS даёт precision@5 3.18% / hit_rate@5 13.36% на common-users
и precision@5 2.58% / hit_rate@5 10.85% на полном скоупе all-users.


In [16]:
def decode_item(item_index):
    """Индекс колонки ALS-матрицы -> имя продукта.

    Колонки матрицы построены по кодам продуктов (1..24), колонка 0 — служебная,
    поэтому products[idx - 1], а не products[idx].
    """
    idx = int(item_index)
    if 1 <= idx <= len(products):
        return products[idx - 1]
    return None


In [17]:
# --- Оценка качества ALS на отложенном периоде ---
K = 5

als_top_k = als_model.recommend(
    user_ids_encoded,
    user_item_matrix_train[user_ids_encoded],
    filter_already_liked_items=True,
    N=K
)

recommendations_df = pd.DataFrame({
    'ncodpers': [inv_user_map[int(user_id)] for user_id in user_ids_encoded],
    'recommended': [
        [name for name in (decode_item(i) for i in items) if name is not None]
        for items in als_top_k[0]
    ],
})

# Релевантные продукты: все покупки клиента в 2016 году — для всех пользователей
# с событиями 2016 года, а не только common (в этом разница скоупов)
relevant_all = (
    common_test
    .groupby('ncodpers')['purchase']
    .apply(lambda codes: {products[int(code) - 1] for code in codes})
)

# Клиенты 2016 года без истории 2015 (cold start): персональных рекомендаций нет
cold_start_users = [int(user) for user in relevant_all.index
                    if int(user) not in set(recommendations_df['ncodpers'])]
eval_df = pd.concat([
    recommendations_df,
    pd.DataFrame({'ncodpers': cold_start_users,
                  'recommended': [[] for _ in cold_start_users]}),
], ignore_index=True)
eval_df['relevant'] = (
    eval_df['ncodpers'].map(relevant_all)
    .apply(lambda items: items if isinstance(items, set) else set())
)
eval_df['hits'] = [
    len(set(recommended) & relevant)
    for recommended, relevant in zip(eval_df['recommended'], eval_df['relevant'])
]
eval_df['relevant_size'] = eval_df['relevant'].apply(len)

os.makedirs('artifacts', exist_ok=True)
als_metric_rows = []
for scope, mask in [('common_users_2015_2016',
                     eval_df['ncodpers'].isin(set(recommendations_df['ncodpers']))),
                    ('all_users_2016', eval_df['ncodpers'].notna())]:
    scope_df = eval_df[mask]
    with_purchases = scope_df['relevant_size'] > 0
    als_metric_rows.append({
        'scope': scope,
        'k': K,
        'users': len(scope_df),
        'precision_at_k': float((scope_df['hits'] / K).mean()),
        'recall_at_k': float(
            (scope_df.loc[with_purchases, 'hits']
             / scope_df.loc[with_purchases, 'relevant_size']).mean()),
        'hit_rate': float((scope_df['hits'] > 0).mean()),
        'relevant_per_user': float(scope_df['relevant_size'].mean()),
    })
als_metrics = pd.DataFrame(als_metric_rows)
als_metrics.to_csv('artifacts/als_metrics.csv', index=False)
print(als_metrics.to_string(index=False))
print()
share_with_purchases = (eval_df['relevant_size'] > 0).mean()
print(f'В оценочном скоупе {share_with_purchases:.2%} клиентов имеют хотя бы одну покупку 2016 года')
print('ALS — самостоятельный baseline top-5 и признак финального классификатора. '
      'Для сравнения финальной модели со случайным бейзлайном смотрите PR-AUC/lift '
      'в итоговом отчёте ниже и в artifacts/classification_report.txt.')


                 scope  k  users  precision_at_k  recall_at_k  hit_rate  relevant_per_user
common_users_2015_2016  5  94091        0.031818     0.095961  0.133647           1.510665
        all_users_2016  5 115867        0.025838     0.077926  0.108530           1.537349

В оценочном скоупе 100.00% клиентов имеют хотя бы одну покупку 2016 года
ALS — самостоятельный baseline top-5 и признак финального классификатора. Для сравнения финальной модели со случайным бейзлайном смотрите PR-AUC/lift в итоговом отчёте ниже и в artifacts/classification_report.txt.


# Теперь делаем рекомендации на основе данных о клиенте

In [18]:
# suppress warnings
warnings.filterwarnings('ignore')

# Фильтрация данных за 2015 год
df2015 = df[df['fecha_dato'] <= '2015-12-31']

# Получаем индексы последних записей для каждого клиента
last_idx = df2015.groupby('ncodpers')['fecha_dato'].idxmax()

# Собираем финальный датасет за один проход.
# fecha_dato среза СОХРАНЯЕМ: по нему строится временное разбиение train/test
# (удаляется после сплита). age/fecha_alta теперь не удаляем здесь — сырые
# значения нужны предобработке (fit считает по ним бины и когорты), а из
# финальной матрицы признаков их отфильтрует BASE_COLUMNS.
train_df = df2015.loc[last_idx].reset_index(drop=True)

train_df.head()


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,ult_fec_cli_1t,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-12-28,15889,F,ES,V,56.0,1995-01-16,0.0,250.0,NaN,...,0,0,0,0,1,1,0,0.0,0.0,0
1,2015-12-28,15890,A,ES,V,63.0,1995-01-16,0.0,251.0,NaN,...,0,1,0,0,1,0,0,1.0,1.0,1
2,2015-08-28,15891,N,ES,H,59.0,2015-07-28,0.0,246.0,2015-08-05,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-12-28,15892,F,ES,H,61.0,1995-01-16,0.0,251.0,NaN,...,0,0,0,1,1,1,0,0.0,0.0,1
4,2015-12-28,15893,N,ES,V,62.0,1997-10-03,0.0,251.0,NaN,...,0,0,0,0,0,1,0,0.0,0.0,0


In [19]:
train_df.to_csv('data/train_final.csv', index=False)


In [20]:
first_buy = filtered_events_test.groupby('ncodpers')['fecha_dato'].idxmin()
first_buy_df = filtered_events_test.loc[first_buy, ['ncodpers', 'purchase']].copy()

# Продукты вне шорт-листа сворачиваем в 'other': иначе в таргете появляются классы
# с support = 1-2 объекта, а метрики по ним ничего не значат
first_buy_df['purchase'] = first_buy_df['purchase'].where(
    first_buy_df['purchase'].isin(target_codes), OTHER_CLASS
)

full_df = pd.merge(train_df, first_buy_df, on='ncodpers', how='left')

# 0 — «в 2016 году клиент ничего не купил»
full_df['purchase'] = full_df['purchase'].fillna(0).astype(int)

# Классы с единичными объектами делают сплит и метрики бессмысленными —
# сворачиваем их в 'other'
MIN_CLASS_SUPPORT = 100
class_counts = full_df['purchase'].value_counts()
rare_classes = class_counts[(class_counts < MIN_CLASS_SUPPORT) & (class_counts.index != 0)].index
if len(rare_classes):
    print('Свернуты в other из-за малого support:',
          {int(code): label_map.get(int(code)) for code in rare_classes})
    full_df['purchase'] = full_df['purchase'].where(
        ~full_df['purchase'].isin(rare_classes), OTHER_CLASS
    )

print('Распределение классов (0 = покупки не было, {} = other):'.format(OTHER_CLASS))
print(full_df['purchase'].map(label_map).value_counts())


Распределение классов (0 = покупки не было, 99 = other):
purchase
no_purchase          833487
ind_recibo_ult1       31659
ind_tjcr_fin_ult1     14378
ind_nomina_ult1       14150
ind_cco_fin_ult1      10432
ind_ecue_fin_ult1      7151
ind_cno_fin_ult1       6168
ind_nom_pens_ult1      3726
ind_reca_fin_ult1      1592
other                  1321
ind_dela_fin_ult1      1163
ind_valo_fin_ult1      1134
ind_ctop_fin_ult1       754
ind_ctpp_fin_ult1       463
Name: count, dtype: int64


In [21]:
full_df = pd.merge(full_df, recommended_tracks_df, on='ncodpers', how='left')

n_without_recs = int(full_df['recommended_product_id'].isna().sum())
print(f'Без персональной ALS-рекомендации: {n_without_recs:,} '
      f'({n_without_recs / len(full_df):.1%}) — для них признак = 0')

full_df['recommended_product_id'] = full_df['recommended_product_id'].fillna('0')


Без персональной ALS-рекомендации: 833,671 (89.9%) — для них признак = 0


Итак, набор данных для обучения модели готов. Целевая переменная — колонка `purchase`:
код первого купленного в 2016 году продукта **из шорт-листа**, `0`, если клиент ничего
не купил, `other` (`99`), если первым куплен редкий продукт.

Почему шорт-лист, а не все 24 продукта: у большинства продуктов доля покупок меньше
процента, и классы по ним состояли бы из одного-двух объектов. Состав шорт-листа и правило
его отбора печатаются в разделе «Целевая переменная», туда же смотрит
`fastapi/preprocessing_params.json` (ключ `label_map`).

### Временное разбиение train/test

Случайное разбиение для этой задачи даёт оптимистичные метрики, поэтому используется
временное разбиение с осмысленной временной осью.

- Разбивать по **дате среза** (последнему `fecha_dato` клиента) нельзя: клиенты с последним
  срезом, скажем, в мае 2015 — это те, кто ушёл из банка до конца года, и «покупок
  в 2016» у них нет априори. Такой train оказывается почти целиком из бывших клиентов
  с таргетом 0 — вырожденное, смещённое разбиение.
- Разбиваем по **дате привлечения клиента (`fecha_alta`)**: train — клиенты, привлечённые
  до cutoff-даты (≈70%), test — более новые клиенты. Сплит по времени привлечения не зависит
  от исхода 2016 года, профили одного клиента не смешиваются, а задача остаётся прежней
  («предсказать первую покупку 2016 года»). По сути это ответ на вопрос «обобщится ли
  модель, обученная на предыдущих клиентских когортах, на новых клиентах».

### Предобработка без утечек

Медианы, моды, квантили клиппинга, возрастные бины, когорты дат и наборы редких
категорий обучаются только на train (`fit_preprocessing_params`) и затем применяются
к test той же функцией `apply_preprocessing`.
Обе части затем трансформируются одной функцией `apply_preprocessing` — тем же
кодом, что использует сервис (`prepare_features`), поэтому train/serve skew
исключён по построению.


In [25]:
# Временное разбиение по дате привлечения клиента (preprocessing.temporal_split):
# train — привлечённые до cutoff, test — новее. По дате среза (fecha_dato) делить
# нельзя: последний срез раного клиента означает отток из банка (см. markdown выше).
# 1) Жёстко приводим к datetime64[ns]
fecha_alta = pd.to_datetime(full_df['fecha_alta'], errors='coerce')
fecha_alta = pd.Series(pd.to_datetime(fecha_alta, errors='coerce'), index=full_df.index)

# 2) Убираем строки, где дата не распарсилась — их всё равно нельзя корректно разбить
valid = fecha_alta.notna()
n_bad = (~valid).sum()
if n_bad:
    print(f'[temporal_split] отбрасываем {n_bad} строк с некорректной fecha_alta')
    full_df = full_df.loc[valid].reset_index(drop=True)
    fecha_alta = fecha_alta.loc[valid].reset_index(drop=True)

assert fecha_alta.dtype == 'datetime64[ns]', fecha_alta.dtype
assert fecha_alta.notna().all()

# 3) Только теперь вызываем разбиение
cutoff, train_mask, test_mask = temporal_split(fecha_alta, test_size=0.3)
cutoff = pd.Timestamp(cutoff)  # на случай, если вернулся np.datetime64/float

train_raw = full_df[train_mask].reset_index(drop=True)
test_raw  = full_df[test_mask].reset_index(drop=True)
# Клиент — одна строка; разбиение гарантирует, что профили одного клиента
# не встречаются в обеих частях (проверяем явно)
assert not set(train_raw['ncodpers']) & set(test_raw['ncodpers']), 'пересечение клиентов'

print(f'cutoff (fecha_alta): {cutoff.date()}')
print(f'train: {len(train_raw):,} строк, fecha_alta '
      f'{train_raw["fecha_alta"].min()} — {train_raw["fecha_alta"].max()}')
print(f'test:  {len(test_raw):,} строк, fecha_alta '
      f'{test_raw["fecha_alta"].min()} — {test_raw["fecha_alta"].max()}')
print(f'доля test: {len(test_raw) / len(full_df):.1%}')

y_train = train_raw['purchase']
y_test = test_raw['purchase']
print('Классы в train:', y_train.value_counts().to_dict())
print('Классы в test:', y_test.value_counts().to_dict())

# fecha_dato и идентификатор в признаки не входят; таргет отделяем от признаков
X_train_raw = train_raw.drop(columns=['purchase', 'fecha_dato', 'ncodpers'])
X_test_raw = test_raw.drop(columns=['purchase', 'fecha_dato', 'ncodpers'])


cutoff (fecha_alta): 2013-10-09
train: 644,744 строк, fecha_alta 1995-01-16 — 2013-10-09
test:  275,674 строк, fecha_alta 2013-10-10 — 2015-12-31
доля test: 30.0%
Классы в train: {0: 577027, 24: 21116, 19: 13311, 22: 10549, 3: 6937, 13: 4419, 5: 3600, 23: 3230, 12: 903, 18: 883, 20: 879, 8: 754, 99: 673, 9: 463}
Классы в test: {0: 249375, 24: 10543, 22: 3600, 3: 3450, 13: 2731, 5: 2566, 19: 1067, 18: 709, 99: 623, 23: 496, 12: 259, 20: 255}


In [26]:
# Статистики предобработки — только на train:
# медианы, моды, возрастные бины, когорты дат, редкие категории, квантили
# клиппинга и эталонные гистограммы дрейфа (для мониторинга в сервисе).
params = fit_preprocessing_params(X_train_raw)

print('Медианы train:', params.medians)
print('Границы клиппинга train:', params.clip_bounds)
print('Возрастные интервалы train:', params.age_intervals)
print('Когорты дат:', {col: len(spec.get('intervals', ()))
                       for col, spec in params.date_cohorts.items()})
print(f'Сколько колонок с редкими категориями: '
      f'{sum(bool(values) for values in params.replacer.values())}')
print('Эталон дрейфа:', sorted(params.drift_reference or {}))


Медианы train: {'age': 43.0, 'antiguedad': 98.0, 'renta': 106413.46500000001}
Границы клиппинга train: {'renta': [28891.3986, 326775.42720000003], 'antiguedad': [22.0, 215.0]}
Возрастные интервалы train: [[2, 25], [26, 34], [35, 40], [41, 44], [45, 47], [48, 51], [52, 56], [57, 65], [66, 163]]
Когорты дат: {'fecha_alta': 12, 'ult_fec_cli_1t': 2}
Сколько колонок с редкими категориями: 11
Эталон дрейфа: ['age', 'antiguedad', 'renta']


In [27]:
# Трансформация обеих частей параметрами train — тем же кодом, что на проде.
# Рекомендация ALS уже подмёржена (recommended_product_id), на проде её
# подставляет lookup по ncodpers — дальше путь общий.
X_train = apply_preprocessing(X_train_raw, params)
X_test = apply_preprocessing(X_test_raw, params)

# Матрица признаков — ровно базовые колонки модели (как в сервисе)
X_train = X_train[BASE_COLUMNS]
X_test = X_test[BASE_COLUMNS]
print(f'train: {X_train.shape}, test: {X_test.shape}')


train: (644744, 42), test: (275674, 42)


### Синтез новых признаков

In [28]:
# Генерация признаков — из общего модуля предобработки:
# обучение и сервис используют одни и те же функции.
# Агрегаты (mean_renta_by_pais_residencia, median_antiguedad_by_segmento и др.)
# считаются только на train и затем применяются к test; те же агрегаты уезжают
# в fastapi/preprocessing_params.json и используются сервисом.
X_train['antiguedad'] = X_train['antiguedad'].astype(int)
X_test['antiguedad'] = X_test['antiguedad'].astype(int)

df_train, aggregates = feature_engineering(X_train)
df_test, _ = feature_engineering(X_test, aggregates)


In [29]:
# Автоматическое определение количественных и категориальных переменных —
# по TRAIN: категориальные — это нечисловые колонки (в них после сворачивания
# редких категорий появляется 'other') и колонки с малым числом уникальных
# значений. Те же списки применяются к test и уезжают в артефакт сервиса.
numeric_features = []
categorical_features = []

for column in df_train.columns:
    unique_values = df_train[column].nunique()
    if unique_values < 25 or not pd.api.types.is_numeric_dtype(df_train[column]):
        categorical_features.append(column)
    else:
        numeric_features.append(column)

print(f'Numeric features ({len(numeric_features)}):', numeric_features)
print(f'Categorical features ({len(categorical_features)}):', categorical_features)

for cat in tqdm.tqdm(categorical_features, desc='Converting categorical features to string'):
    df_train[cat] = df_train[cat].astype('str')
    df_test[cat] = df_test[cat].astype('str')

# Контроль остаточных пропусков (после fillna/агрегатов их быть не должно)
n_nan = int(df_train.isna().sum().sum() + df_test.isna().sum().sum())
print(f'NaN в матрицах признаков: {n_nan}')


Numeric features (13): ['antiguedad', 'renta', 'antiguedad + renta', 'antiguedad / renta', 'renta / antiguedad', 'antiguedad * renta', 'NATURAL_LOGARITHM(antiguedad)', 'NATURAL_LOGARITHM(renta)', 'SQUARE_ROOT(antiguedad)', 'SQUARE_ROOT(renta)', 'renta_antiguedad_ratio', 'log_renta', 'renta_vs_country_mean']
Categorical features (44): ['ind_empleado', 'pais_residencia', 'sexo', 'ind_nuevo', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'conyuemp', 'canal_entrada', 'indfall', 'cod_prov', 'ind_actividad_cliente', 'segmento', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 

Converting categorical features to string: 100%|██████████| 44/44 [00:04<00:00, 10.00it/s]


NaN в матрицах признаков: 0


In [30]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [31]:
load_dotenv()
TRACKING_SERVER_HOST = os.getenv('MLFLOW_TRACKING_HOST', '127.0.0.1')
TRACKING_SERVER_PORT = int(os.getenv('MLFLOW_TRACKING_PORT', '5000'))

EXPERIMENT_NAME = "RecSys_Modeling"
RUN_NAME = "fit"

FS_ASSETS = 'modeling'
os.makedirs(FS_ASSETS, exist_ok=True)

# Локальный tracking-сервер MLflow
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

mlflow.set_experiment(EXPERIMENT_NAME)


2026/09/17 10:46:08 INFO mlflow.tracking.fluent: Experiment with name 'RecSys_Modeling' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///E:/Data/OTUS/homework/otus_prj_bank_rec_sys/mlruns_artifacts/2', creation_time=1789631167788, experiment_id='2', last_update_time=1789631167788, lifecycle_stage='active', name='RecSys_Modeling', tags={}>

In [40]:
def cross_val_roc_auc(pipeline, X_train, y_train, cv):
    """ROC-AUC по фолдам кросс-валидации — тестовая выборка не используется."""
    scores = []
    for train_index, valid_index in cv.split(X_train, y_train):
        fold_model = clone(pipeline).fit(X_train.iloc[train_index], y_train.iloc[train_index])
        scores.append(safe_macro_roc_auc(
            y_train.iloc[valid_index].to_numpy(),
            fold_model.predict_proba(X_train.iloc[valid_index]),
            list(fold_model.classes_),
        ))
    return np.asarray(scores, dtype=float)


def make_objective(pipeline, X_train, y_train, cv):
    """Objective для Optuna: ROC-AUC по кросс-валидации на train.

    Тестовая выборка в подборе гиперпараметров не участвует:
    иначе метрика на тесте — это максимум из перебора, а не честная оценка.
    """

    def objective(trial):
        # Извлечение параметров
        params_trial = {
            'classifier__n_estimators': trial.suggest_int('classifier__n_estimators', 5, 50),
            'classifier__max_depth': trial.suggest_categorical(
                'classifier__max_depth', [1, 5, 10, 30, None]
            ),
            'classifier__min_samples_split': trial.suggest_int(
                'classifier__min_samples_split', 2, 20
            ),
        }
        pipeline.set_params(**params_trial)

        # Оценка — только по фолдам внутри train
        scores = cross_val_roc_auc(pipeline, X_train, y_train, cv)
        trial.set_user_attr('cv_std', float(np.nanstd(scores)))
        return float(np.nanmean(scores))

    return objective


In [41]:
def safe_macro_roc_auc(y_true, y_proba, classes):
    """macro ROC-AUC (ovr) с защитой от классов, которых нет в модели,
    и от не-нормированных вероятностей."""
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    classes = list(classes)

    # Ренормировка на всякий случай
    s = y_proba.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    y_proba = y_proba / s

    # Оставляем только те классы, которые есть и в y_true, и в y_proba
    present = [c for c in classes if (y_true == c).any()]
    if not present:
        return float('nan')

    idx = [classes.index(c) for c in present]
    y_proba_sub = y_proba[:, idx]
    y_true_sub  = np.array([present.index(v) for v in y_true
                            if v in present])
    y_true_mask = np.isin(y_true, present)
    y_proba_sub = y_proba_sub[y_true_mask]

    if len(present) == 2:
        return float(roc_auc_score(y_true_sub, y_proba_sub[:, 1]))
    return float(roc_auc_score(
        label_binarize(y_true_sub, classes=list(range(len(present)))),
        y_proba_sub,
        multi_class='ovr',
        average='macro',
    ))

In [42]:
with mlflow.start_run(run_name=RUN_NAME):
    # Разбиение уже выполнено выше — временное, по дате привлечения клиента
    X_train_ml, X_test_ml = df_train, df_test
    logger.info("Data loaded and preprocessed.")
    print(f'train: {X_train_ml.shape}, test: {X_test_ml.shape}')

    # Параметры разбиения и обучения — в MLflow (воспроизводимость)
    mlflow.log_param('split_type', 'temporal_fecha_alta')
    mlflow.log_param('split_cutoff', str(cutoff.date()))
    mlflow.log_param('split_test_size', 0.3)
    mlflow.log_param('preprocessing_fit', 'train_only')

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    logger.info("Preprocessor initialized.")

    # Пайплайн с учетом дисбаланса классов
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectFromModel(
            RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=0)
        )),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=0))
    ])
    logger.info("Pipeline created with class weight handling.")

    # Поиск гиперпараметров с Optuna: скоринг по кросс-валидации на train,
    # тестовая выборка используется один раз — для финальной оценки
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(pipeline, X_train_ml, y_train, cv), n_trials=20, timeout=1200)

    completed_trials = [trial for trial in study.trials
                        if trial.state == optuna.trial.TrialState.COMPLETE]
    if not completed_trials:
        logger.error("Optuna failed to complete any trials within time limit.")
        raise ValueError("No Optuna trials completed")

    # Логирование лучших параметров
    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.log_metric('cv_roc_auc_mean', study.best_value)
    mlflow.log_metric('cv_roc_auc_std', study.best_trial.user_attrs.get('cv_std', float('nan')))
    mlflow.log_metric('optuna_trials', len(study.trials))
    print(f'Optuna: {len(study.trials)} триалов, лучший CV ROC-AUC = {study.best_value:.4f}')
    print('Лучшие параметры:', best_params)

    # Обучение финальной модели на всем train
    final_model = pipeline.set_params(**best_params).fit(X_train_ml, y_train)
    logger.info("Final model trained with best parameters.")

    # --- Оценка: тест трогаем один раз, после выбора гиперпараметров --------
    y_pred = final_model.predict(X_test_ml)
    y_proba = final_model.predict_proba(X_test_ml)
    classes = [int(cls) for cls in final_model.named_steps['classifier'].classes_]

    classes = [int(c) for c in final_model.named_steps['classifier'].classes_]
    y_test_arr = np.asarray(y_test)

    # 1) Классы, которых не было в трейне
    unseen = set(np.unique(y_test_arr)) - set(classes)
    if unseen:
        print(f'[WARN] в y_test есть классы, не встречавшиеся в train: {unseen}')

    # 2) NaN/inf в признаках
    if np.isnan(X_test_ml.select_dtypes(include=[np.number])).any().any():
        print('[WARN] в X_test_ml есть NaN')

    # 3) Сумма вероятностей
    row_sums = np.asarray(y_proba).sum(axis=1)
    print(f'y_proba sum: min={row_sums.min():.6f}, max={row_sums.max():.6f}, '
      f'mean={row_sums.mean():.6f}, any_nan={np.isnan(row_sums).any()}')
    classes = [int(c) for c in final_model.named_steps['classifier'].classes_]
    y_test_arr = np.asarray(y_test)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    # --- Защита 1: если y_test содержит классы, которых нет в модели --------
    # Их нельзя корректно ни предсказать, ни оценить per-class ROC-AUC.
    # Измеряем их долю — если она значима, это сигнал, что модель недоучена.
    mask_known = np.isin(y_test_arr, classes)
    n_unknown = (~mask_known).sum()
    if n_unknown:
        logger.warning(
            "y_test содержит %d строк (%.2f%%) с классами вне train: %s. "
            "Метрики считаем на пересечении классов.",
            n_unknown, 100 * n_unknown / len(y_test_arr),
            sorted(set(np.unique(y_test_arr)) - set(classes)),
        )

    # --- Защита 2: NaN/inf в признаках — не падаем, а сообщаем и чистим ------
    if not np.isfinite(y_proba).all():
        logger.warning("В y_proba есть NaN/inf — заменяем на 0 и ренормируем строки")
        y_proba = np.nan_to_num(y_proba, nan=0.0, posinf=0.0, neginf=0.0)

    # --- Защита 3: ренормировка (на случай float32/численного дрейфа) --------
    row_sums = y_proba.sum(axis=1, keepdims=True)
    # защита от деления на 0
    row_sums[row_sums == 0] = 1.0
    y_proba = y_proba / row_sums

    # --- Защита 4: работаем только с известными классами ---------------------
    # Для метрик берём подвыборку y_test/y_pred по известным классам
    y_test_eval = y_test_arr
    y_pred_eval = y_pred

    metrics = {
        'roc_auc_ovr_macro': safe_macro_roc_auc(y_test_eval, y_proba, classes),
        'accuracy':          accuracy_score(y_test_eval, y_pred_eval),
        'precision_weighted': precision_score(y_test_eval, y_pred_eval, average='weighted',
                                            zero_division=0),
        'recall_weighted':    recall_score(y_test_eval, y_pred_eval, average='weighted',
                                        zero_division=0),
        'f1_weighted':        f1_score(y_test_eval, y_pred_eval, average='weighted',
                                    zero_division=0),
        'precision_macro':    precision_score(y_test_eval, y_pred_eval, average='macro',
                                            zero_division=0),
        'recall_macro':       recall_score(y_test_eval, y_pred_eval, average='macro',
                                        zero_division=0),
        'f1_macro':           f1_score(y_test_eval, y_pred_eval, average='macro',
                                    zero_division=0),
    }

    # PR-AUC (average precision) по каждому классу: для сильного дисбаланса
    # это честнее ROC-AUC, а базовый уровень случайного ранжирования = доля класса
    pr_auc_rows = []
    for index, cls in enumerate(classes):
        y_binary = (y_test.to_numpy() == cls).astype(int)
        if y_binary.sum() == 0:
            continue
        pr_auc_rows.append({
            'class': cls,
            'product': label_map.get(cls, str(cls)),
            'support': int(y_binary.sum()),
            'share': float(y_binary.mean()),
            'pr_auc': float(average_precision_score(y_binary, y_proba[:, index])),
            'roc_auc_ovr': float(roc_auc_score(y_binary, y_proba[:, index])),
        })
    pr_auc_df = pd.DataFrame(pr_auc_rows).sort_values('share', ascending=False)
    metrics['pr_auc_macro'] = float(pr_auc_df['pr_auc'].mean())

    # Бейзлайны для интерпретации качества рекомендаций.
    no_purchase_share = float(
        pr_auc_df.loc[pr_auc_df['class'] == 0, 'share'].iloc[0]
    )
    product_pr_auc_df = pr_auc_df[
        (~pr_auc_df['class'].isin([0, OTHER_CLASS])) & (pr_auc_df['support'] > 0)
    ]
    product_mean_pr_auc = float(product_pr_auc_df['pr_auc'].mean())
    product_random_pr_auc = float(product_pr_auc_df['share'].mean())
    product_pr_auc_lift = product_mean_pr_auc / product_random_pr_auc
    baseline_metrics = {
        'no_purchase_accuracy': no_purchase_share,
        'accuracy_lift_pp': (metrics['accuracy'] - no_purchase_share) * 100,
        'product_classes_mean_pr_auc': product_mean_pr_auc,
        'product_classes_mean_random_pr_auc': product_random_pr_auc,
        'product_classes_pr_auc_lift': product_pr_auc_lift,
    }

    mlflow.log_metrics(metrics)
    mlflow.log_metrics({f'baseline_{key}': value for key, value in baseline_metrics.items()})
    for row in pr_auc_rows:
        mlflow.log_metric(f"pr_auc_class_{row['class']}", row['pr_auc'])
        mlflow.log_metric(f"support_class_{row['class']}", row['support'])
    logger.info("Model metrics logged.")

    mlflow.sklearn.log_model(final_model, "model")
    logger.info("Best model logged to MLflow.")
    training_run_id = mlflow.active_run().info.run_id
    print(f'MLflow run_id: {training_run_id}')

    # Отчёт о качестве: per-class таблица + macro + PR-AUC + сравнение с бейзлайном
    report = classification_report(y_test, y_pred, digits=4, zero_division=0)
    with open('artifacts/classification_report.txt', 'w', encoding='utf-8') as f:
        f.write('Отчёт о качестве последнего обучающего запуска (modeling.ipynb)\n')
        f.write(f'Целевая переменная: первый купленный продукт 2016 года из шорт-листа; '
                f'классы: {label_map}\n')
        f.write('Разбиение: временное по дате привлечения клиента (fecha_alta), '
                f'cutoff {cutoff.date()}; '
                f'train: {X_train_ml.shape}, test: {X_test_ml.shape}\n')
        f.write('Статистики предобработки и агрегаты обучены только на train; '
                'гиперпараметры выбраны по CV на train, тест использован один раз\n\n')
        f.write('Взвешенные метрики (доминирует класс "покупки не было"):\n')
        for name, value in metrics.items():
            f.write(f'  {name}: {value:.4f}\n')
        f.write('\nСравнение с бейзлайном:\n')
        f.write(
            f'  accuracy бейзлайна no_purchase: '
            f'{baseline_metrics["no_purchase_accuracy"]:.4f} '
            '(если всегда предсказывать «покупки нет»)\n'
        )
        f.write(
            f'  текущая accuracy: {metrics["accuracy"]:.4f} '
            f'({baseline_metrics["accuracy_lift_pp"]:+.2f} п.п. к no_purchase-бейзлайну)\n'
        )
        f.write(
            f'  средний PR-AUC продуктовых классов: '
            f'{baseline_metrics["product_classes_mean_pr_auc"]:.4f}\n'
        )
        f.write(
            f'  средний случайный PR-AUC продуктовых классов: '
            f'{baseline_metrics["product_classes_mean_random_pr_auc"]:.4f}\n'
        )
        f.write(
            f'  lift PR-AUC продуктовых классов: '
            f'{baseline_metrics["product_classes_pr_auc_lift"]:.1f}x\n'
        )
        f.write('  Примечание: PR-AUC считается честным сравнением для рекомендаций;\n')
        f.write('  случайный PR-AUC равен доле соответствующего класса в test.\n')
        f.write('\n' + report)
        f.write('\nPR-AUC по классам (случайное ранжирование даёт PR-AUC = доля класса):\n')
        f.write(pr_auc_df.to_string(index=False) + '\n')
    mlflow.log_artifact('artifacts/classification_report.txt')
    logger.info("Classification report logged to MLflow.")
    print('\n' + report)
    print(pr_auc_df.to_string(index=False))

    # Feature importance
    sup = final_model.named_steps['feature_selection'].get_support()
    fname = final_model.named_steps['preprocessor'].get_feature_names_out()
    fi_df = pd.DataFrame({'feature': fname, 'support': sup})
    fi_df = fi_df[fi_df['support']]
    fi_df['importance'] = final_model.named_steps['classifier'].feature_importances_
    fi_df = fi_df.sort_values(by='importance', ascending=False)
    fi_df.to_csv('artifacts/feature_importances.csv', index=False)
    mlflow.log_artifact('artifacts/feature_importances.csv')
    logger.info("Feature importances logged to MLflow.")

    if os.path.exists('artifacts/als_metrics.csv'):
        mlflow.log_artifact('artifacts/als_metrics.csv')

    logger.info(f"Метрики на тесте: ROC-AUC (ovr macro) = {metrics['roc_auc_ovr_macro']:.4f}, "
                f"F1 macro = {metrics['f1_macro']:.4f}, "
                f"PR-AUC macro = {metrics['pr_auc_macro']:.4f}")
    logger.info("Experiment completed successfully!")


INFO:__main__:Data loaded and preprocessed.


train: (644744, 57), test: (275674, 57)


INFO:__main__:Preprocessor initialized.
INFO:__main__:Pipeline created with class weight handling.
[I 2026-09-17 12:48:18,739] A new study created in memory with name: no-name-af31507f-0200-4a2e-9243-3133e59e0b10
[I 2026-09-17 12:50:49,987] Trial 0 finished with value: 0.89056376045531 and parameters: {'classifier__n_estimators': 22, 'classifier__max_depth': 1, 'classifier__min_samples_split': 3}. Best is trial 0 with value: 0.89056376045531.
[I 2026-09-17 12:54:48,494] Trial 1 finished with value: 0.8915545968982469 and parameters: {'classifier__n_estimators': 44, 'classifier__max_depth': 30, 'classifier__min_samples_split': 6}. Best is trial 1 with value: 0.8915545968982469.
[I 2026-09-17 12:57:35,146] Trial 2 finished with value: 0.9551547104977249 and parameters: {'classifier__n_estimators': 13, 'classifier__max_depth': 10, 'classifier__min_samples_split': 13}. Best is trial 2 with value: 0.9551547104977249.
[I 2026-09-17 13:00:22,152] Trial 3 finished with value: 0.856956277861095

Optuna: 6 триалов, лучший CV ROC-AUC = 0.9622
Лучшие параметры: {'classifier__n_estimators': 42, 'classifier__max_depth': 10, 'classifier__min_samples_split': 11}


INFO:__main__:Final model trained with best parameters.


y_proba sum: min=1.000000, max=1.000000, mean=1.000000, any_nan=False


INFO:__main__:Model metrics logged.
2026/09/17 13:15:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
INFO:__main__:Best model logged to MLflow.


MLflow run_id: 01ba6df2ddd0498590fcf00ef77f77d0


INFO:__main__:Classification report logged to MLflow.
INFO:__main__:Feature importances logged to MLflow.
INFO:__main__:Метрики на тесте: ROC-AUC (ovr macro) = 0.9594, F1 macro = 0.2816, PR-AUC macro = 0.3437
INFO:__main__:Experiment completed successfully!



              precision    recall  f1-score   support

           0     0.9997    0.9999    0.9998    249375
           3     0.7190    0.3635    0.4829      3450
           5     0.3199    0.1952    0.2425      2566
           8     0.0000    0.0000    0.0000         0
           9     0.0000    0.0000    0.0000         0
          12     0.0343    0.0656    0.0451       259
          13     0.2638    0.3482    0.3002      2731
          18     0.0392    0.0423    0.0407       709
          19     0.0000    0.0000    0.0000      1067
          20     0.0285    0.3137    0.0522       255
          22     0.4845    0.6642    0.5603      3600
          23     0.8063    0.7722    0.7889       496
          24     0.7688    0.1940    0.3098     10543
          99     0.0651    0.7480    0.1197       623

    accuracy                         0.9340    275674
   macro avg     0.3235    0.3362    0.2816    275674
weighted avg     0.9564    0.9340    0.9367    275674

 class           product

Вывод: модель обучена, метрики записаны в MLflow, а текстовые отчёты сохранены в `artifacts/`.


### Артефакт параметров предобработки

Всё, что нужно для предсказания в сервисе (медианы, моды, границы клиппинга,
возрастные бины, когорты дат, агрегаты, словарь редких категорий, расшифровка
классов и эталонные гистограммы дрейфа), сохраняется в
`fastapi/preprocessing_params.json`.

Сервис `app1.py` читает этот файл, поэтому предобработка на проде совпадает с
обучением. Статистики обучены только на train; в блоке `split` фиксируются тип
разбиения и cutoff-дата.


In [ ]:
def json_converter(obj):
    """Конвертер numpy/pandas-типов в JSON для артефактов сервиса."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, pd.Timestamp):
        return obj.strftime('%Y-%m-%d')
    if isinstance(obj, pd.Interval):
        return [float(obj.left), float(obj.right)]
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")


def save_json(path, payload):
    """Сохраняет артефакт предобработки для микросервиса."""
    with open(path, 'w', encoding='utf-8') as file:
        json.dump(payload, file, indent=2, ensure_ascii=False, default=json_converter)
    print(f'Сохранено: {path}')


preprocessing_params = {
    'created_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'dataset': 'data/train_ver2.csv',
    'target': {
        'definition': 'первый продукт, купленный клиентом в 2016 году, из шорт-листа',
        'shortlist_rule': f'доля покупок 2015 года >= {MIN_PURCHASE_SHARE:.0%}',
        'classes': {str(code): name for code, name in label_map.items()},
    },
    'split': {
        'type': 'temporal_fecha_alta',
        'cutoff': str(cutoff.date()),
        'test_size': 0.3,
        'definition': 'train: клиенты, привлечённые до cutoff (fecha_alta); test: новее',
        'fit': 'статистики предобработки и агрегаты обучены только на train',
    },
    'medians': params.medians,
    'modes': params.modes,
    'clip_bounds': params.clip_bounds,
    'age_intervals': params.age_intervals,
    'date_cohorts': params.date_cohorts,
    'aggregates': aggregates,
    'replacer': params.replacer,
    'drift_reference': params.drift_reference,
    'label_map': {str(code): name for code, name in label_map.items()},
    'feature_columns': {
        'numeric': numeric_features,
        'categorical': categorical_features,
    },
    'notes': [
        'Файл содержит параметры предобработки для сервиса; при полном запуске '
        'modeling.ipynb секции split и drift_reference пересоздаются автоматически.',
        'Если drift_reference равен null, GET /drift работает в режиме no_reference.',
    ],
}

save_json('fastapi/preprocessing_params.json', preprocessing_params)
save_json('fastapi/replacer.json', params.replacer)
print('Классы модели:', preprocessing_params['label_map'])
print('Агрегаты для сервиса:', list(aggregates))
print('Эталон дрейфа:', sorted(params.drift_reference or {}))


In [ ]:
# Канонический экспорт модели для микросервиса:
# sklearn-пайплайн целиком — сервис грузит его через joblib.load.
# Рядом пишем fastapi/model_version.json (run id, дата, метрики, классы,
# схема разбиения), чтобы версия модели была зафиксирована, а не только в MLflow.
joblib.dump(final_model, 'fastapi/saved_model.pkl')
print('Модель сохранена: fastapi/saved_model.pkl')

als_metrics_payload = {'k': K}
if 'als_metrics' in globals():
    for row in als_metrics.to_dict(orient='records'):
        scope = row.pop('scope')
        row.pop('k', None)
        als_metrics_payload[scope] = row

model_version = {
    'created_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'model_path': 'fastapi/saved_model.pkl',
    'mlflow_run_id': training_run_id if 'training_run_id' in dir() else None,
    'mlflow_run_name': RUN_NAME,
    'experiment': EXPERIMENT_NAME,
    'dataset': 'data/train_ver2.csv',
    'target': preprocessing_params['target'],
    'split': preprocessing_params['split'],
    'train_shape': list(X_train_ml.shape),
    'test_shape': list(X_test_ml.shape),
    'best_params': best_params,
    'cv_roc_auc_mean': float(study.best_value),
    'metrics': {name: round(float(value), 4)
                for name, value in {**metrics, 'optuna_trials': len(study.trials)}.items()},
    'baseline': {name: round(float(value), 4) for name, value in baseline_metrics.items()},
    'als_metrics': als_metrics_payload,
    'label_map': {str(code): name for code, name in label_map.items()},
    'notes': [
        'Числа синхронизированы с выводами modeling.ipynb и artifacts/classification_report.txt.',
        'fastapi/saved_model.pkl не хранится в Git из-за размера и создаётся запуском modeling.ipynb.',
    ],
}
save_json('fastapi/model_version.json', model_version)
print('Версия зафиксирована: fastapi/model_version.json',
      f"(run_id={model_version['mlflow_run_id']})")


# 📊 Анализ метрик рекомендательной системы

## Методология оценки

- Train/test-разбиение временное по `fecha_alta`: train — ранние когорты клиентов,
  test — более новые. Cutoff последнего запуска: `2013-10-09`.
- Статистики предобработки и агрегаты обучены только на train.
- Тестовая выборка используется один раз — после выбора гиперпараметров по CV на train.
- Для рекомендаций главные метрики — per-class PR-AUC, lift относительно случайного
  бейзлайна и macro-метрики. Accuracy нужна только как дополнительный sanity-check,
  потому что класс `no_purchase` занимает 90.46% test.

## Итоги последнего запуска

| Метрика | Значение |
|---|---:|
| Train / test | 644 744 / 275 674 объектов |
| CV ROC-AUC macro | 0.9622 |
| ROC-AUC macro OVR на test | 0.9594 |
| Accuracy | 0.9340 |
| Precision macro | 0.3235 |
| Recall macro | 0.3362 |
| F1 macro | 0.2816 |
| PR-AUC macro | 0.3437 |

## Сравнение с бейзлайном

- Бейзлайн `always no_purchase` даёт accuracy около 90.46%; модель даёт 93.40%,
  то есть выигрывает +2.94 п.п. Но это не основная бизнес-метрика.
- Для PR-AUC случайный бейзлайн равен доле класса. По продуктовым классам с ненулевым
  support средний PR-AUC модели равен 0.296 против средней доли 0.0093 — примерно
  **31.8× лучше случайного ранжирования**.
- По массовым продуктам lift устойчиво двузначный: `ind_recibo_ult1` — 15.3×,
  `ind_cco_fin_ult1` — 40.1×, `ind_nomina_ult1` — 31.7×, `ind_ecue_fin_ult1` — 29.9×.

## Интерпретация

Модель хорошо отделяет клиентов с повышенной вероятностью покупки от случайного потока,
но задача остаётся сложной из-за сильного дисбаланса и большого числа клиентов без
покупки. Для бизнес-использования нужно смотреть top-k рекомендации и lift по каждому
продукту; редкие классы с маленьким support следует перепроверять на новых периодах.
